# ISM 6564 · Week 6 · Workflow example: from a business problem to a working tool

**ISM 6564 — Text Analytics · Fall 2026 · Week 6: Prompting, Frontier APIs & Fine-Tuning**

## About this notebook

- **Optional and ungraded.** It is a worked example of the whole route the lecture described, end to
  end, so you can see how the pieces fit before you need them for your project.
- **The route, in three parts:**

| Part | Where | What happens |
|---|---|---|
| **A** | Google Colab (free T4 GPU) | frame the business problem → find a suitable model on Hugging Face → measure the base model → fine-tune it with LoRA → measure again → **download the adapter** (about 35 MB) |
| **B** | your own computer | turn the adapter into an Ollama model with `ism6564-week06-export-to-ollama.ipynb` (Canvas) |
| **C** | your own computer | a small **Streamlit** page (`ism6564-week06-workflow-app.py`, Canvas) that calls your model through Ollama's OpenAI-style API and checks every reply against the contract |

- **Why split it this way.** The GPU work happens where the GPU is. Your laptop only has to run
  Ollama and a small web page; nothing on it needs PyTorch, and the customer's text never leaves it.
- **Time.** Part A: about 10 minutes on a T4 (2–3 minutes of it training). Part B: about two minutes.
  Part C: seconds.
- **Setup.** In Colab: *Runtime → Change runtime type → T4 GPU*, then *Run all*. On your computer:
  the course environment plus `pip install streamlit openai pydantic`, and Ollama installed
  ([ollama.com/download](https://ollama.com/download)). As always, your machine is your
  responsibility; the setup guide on Canvas has the details.


---

# Part A — on Colab

## A.1 — The business problem

**Northwind Outfitters' support lead** wants a tool a team member can paste a customer email into and
get back the triage object the rest of the support system uses: which queue, how urgent, whether to
page a person, and the order number. Four constraints, written down *before* any model is chosen,
because they decide the model:

1. **Customer text may not leave the company** (no rented API for this tool) → a **local** model.
2. It has to run on **ordinary office hardware** → a **small** model (about a billion numbers or fewer).
3. It must produce **Northwind's own codes and policy** reliably, which no public model was trained on
   → **fine-tune for form and behaviour** (the lecture's rule), on Northwind's labelled tickets.
4. The team must be able to **measure** it before trusting it → the same held-out tickets and the same
   scorer as the lecture.

Notice what is *not* on the list: knowing where a parcel is. That is a fact, and facts are Week 7.


In [ ]:
# --- setup (Colab or your own machine) ---------------------------------------------------------------
import importlib.util, subprocess, sys
for mod, pkg in {"peft": "peft", "pydantic": "pydantic", "huggingface_hub": "huggingface_hub"}.items():
    if importlib.util.find_spec(mod) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

import json, os, random, re, shutil, time
from typing import Literal
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")
import numpy as np, pandas as pd, torch
from pydantic import BaseModel, Field, StrictBool, ValidationError

IN_COLAB = "google.colab" in sys.modules
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
SEED = 6564
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print(f"Colab: {IN_COLAB} · device: {device}" + ("" if device.type != "cpu" else
      "   <- on Colab choose Runtime > Change runtime type > T4 GPU; training on a CPU takes far longer"))

# the data, from the course's public data repository (the same tickets as the lecture and Assignment 6)
BASE_URL = ("https://raw.githubusercontent.com/prof-tcsmith/6564F26-DATA/main/"
            "week-06-prompting-apis-finetuning/data")
train = pd.read_csv(f"{BASE_URL}/support_tickets_train.csv", dtype=str).fillna("")
test = pd.read_csv(f"{BASE_URL}/support_tickets_test.csv", dtype=str).fillna("")
EVAL = test.iloc[40:80].reset_index(drop=True)          # 40 held-out tickets, phrasings never used in training
CODES = ["SHIP-DELAY", "RET-RMA", "BILL-DISP", "PROD-DEF", "ACCT-SEC", "PROD-INFO"]
print(f"{len(train)} training tickets · {len(EVAL)} evaluation tickets")


## A.2 — Find a model on Hugging Face

The Hub has over a million models. The constraints above turn the search into a short checklist:

| Look for | Why | Where on the Hub |
|---|---|---|
| **task = text generation, an *instruct* / *chat* version** | it must follow instructions and use a chat template (Week 5) | the *Text Generation* filter; names ending `-Instruct`, `-it`, `-chat` |
| **size ≈ 0.5–1.5 billion numbers** | constraint 2: office hardware, and a fine-tune that fits on Colab's T4 | the parameter count on the model card / in the *safetensors* metadata |
| **a permissive licence, not gated** | a gated model needs an account and an accepted agreement on every machine; a restrictive licence may forbid the use | the *license* tag; the "gated" notice at the top of the card |
| **a family Ollama can run** | Part B has to serve it | Ollama's model library lists the families it supports |
| **recent, widely downloaded** | more downloads means more people found the bugs first | the downloads counter; the last-updated date |

The cell asks the Hub for the facts about five plausible candidates. (It needs the internet; no
account or token is required for public models.)


In [ ]:
from huggingface_hub import HfApi

CANDIDATES = ["Qwen/Qwen2.5-0.5B-Instruct", "Qwen/Qwen2.5-1.5B-Instruct", "HuggingFaceTB/SmolLM2-360M-Instruct",
              "meta-llama/Llama-3.2-1B-Instruct", "google/gemma-3-1b-it"]
api, rows = HfApi(), []
for repo in CANDIDATES:
    try:
        info = api.model_info(repo)
        params = info.safetensors.total if info.safetensors else None
        rows.append({"model": repo, "params (M)": round(params / 1e6) if params else "?",
                     "license": (info.card_data or {}).get("license", "?"), "gated": bool(info.gated),
                     "downloads / month": info.downloads, "task": info.pipeline_tag})
    except Exception as e:                       # a private or removed repo, or no internet
        rows.append({"model": repo, "params (M)": "?", "license": "?", "gated": "?", "downloads / month": "?", "task": type(e).__name__})
pd.DataFrame(rows)


**Reading the table.** The two gated models need an account and an accepted licence on every machine
that loads them, and Gemma's licence carries use restrictions; both are a poor fit for a tool the whole
team runs. SmolLM2-360M is small and open but weaker at following a contract. **`Qwen/Qwen2.5-0.5B-Instruct`**
is open (Apache-2.0), ungated, half a billion numbers (about 1 GB to download, 2 GB in memory), follows a
chat template, and its family runs in Ollama. The 1.5B sibling is the lecture's model and would score
higher, at three times the size; for an office laptop the 0.5B is the constraint-respecting choice, and
Part A will *measure* whether it is good enough after fine-tuning rather than assume it.


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL_ID); tok.padding_side = "left"
model = AutoModelForCausalLM.from_pretrained(MODEL_ID).to(dtype=torch.float32).to(device)   # full precision (lecture)
n = sum(p.numel() for p in model.parameters())
print(f"{MODEL_ID}: {n/1e6:.0f} M numbers, {n*4/1e9:.1f} GB at full precision, context window {model.config.max_position_embeddings:,} tokens")


## A.3 — The contract, the prompt, and a baseline

The contract is the lecture's hard guardrail (Block 4). The prompt is the **minimal** one the fine-tune
will be trained with: it names the fields and the legal codes but explains nothing, because the
meanings and the policy are going into the weights. First, how does the base model do with only that?


In [ ]:
class Triage(BaseModel):
    route_to: Literal["SHIP-DELAY", "RET-RMA", "BILL-DISP", "PROD-DEF", "ACCT-SEC", "PROD-INFO"]
    urgency: Literal["low", "normal", "high"]
    needs_human: StrictBool                      # a real true/false only: the text "no" is rejected
    order_id: str = Field(default="", pattern=r"^(NW-\d{5})?$")

MINIMAL_SYSTEM = ("You triage support tickets for Northwind Outfitters.\n"
                  'Reply with JSON: {"route_to": one of ' + "|".join(CODES) +
                  ', "urgency": low|normal|high, "needs_human": true|false, "order_id": string or ""}')

def messages(ticket): return [{"role": "system", "content": MINIMAL_SYSTEM},
                              {"role": "user", "content": f"<ticket>\n{ticket}\n</ticket>"}]
def chat(m): return tok.apply_chat_template(m, tokenize=False, add_generation_prompt=True)
def gold(row): return {"route_to": row.category, "urgency": row.urgency, "needs_human": row.needs_human == "true", "order_id": row.order_id}

@torch.no_grad()
def generate(m, tickets, batch=8, max_new=60):
    out = []
    for i in range(0, len(tickets), batch):
        enc = tok([chat(messages(t)) for t in tickets[i:i + batch]], return_tensors="pt", padding=True).to(device)
        g = m.generate(**enc, max_new_tokens=max_new, do_sample=False, temperature=None, top_p=None, top_k=None, pad_token_id=tok.pad_token_id)
        out += [tok.decode(x[enc["input_ids"].shape[1]:], skip_special_tokens=True).strip() for x in g]
    return out

def check(raw):
    m = re.search(r"\{.*?\}", raw, re.S)
    if not m: return None
    try: return Triage.model_validate_json(m.group(0))
    except ValidationError: return None

def score(raws, name):
    objs = [check(r) for r in raws]
    valid = sum(o is not None for o in objs)
    right = {f: 0 for f in ["route_to", "urgency", "needs_human", "order_id"]}; exact = 0
    for o, row in zip(objs, EVAL.itertuples()):
        if o is None: continue
        g = gold(row); d = o.model_dump(); ok = [d[f] == g[f] for f in right]
        for f, k in zip(right, ok): right[f] += k
        exact += all(ok)
    print(f"{name:<28} valid {valid:>2}/{len(EVAL)}   " + "  ".join(f"{f} {right[f]:>2}" for f in right) + f"   ALL FOUR RIGHT {exact:>2}/{len(EVAL)}")
    return exact

t0 = time.time()
base_raw = generate(model, list(EVAL.text))
base_exact = score(base_raw, "base model + minimal prompt")
print(f"({time.time() - t0:.0f} s)   first reply: {base_raw[0][:120]!r}")


The base model with the minimal prompt is the honest starting point: it has never seen Northwind's
codes or policy, so most of what it gets right is guessed from the words. Now teach it.

## A.4 — Fine-tune with LoRA

Exactly the lecture's recipe (Block 5): freeze the model, learn two thin tables in every weight
table (`r = 16`), train on the 720 labelled tickets with the *answer* graded and the prompt masked
with `-100`, one pass, batch 8. Gradient checkpointing keeps the memory inside a T4. About two to
three minutes on a T4; longer on a laptop GPU; a long time on a CPU.


In [ ]:
from peft import LoraConfig, get_peft_model

def example(row, maxlen=288):
    p = tok(chat(messages(row.text)), add_special_tokens=False)["input_ids"]
    a = tok(json.dumps(gold(row)) + tok.eos_token, add_special_tokens=False)["input_ids"]
    return (p + a)[:maxlen], ([-100] * len(p) + a)[:maxlen]          # -100: the prompt is not graded

cfg = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                 target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
peft_model = get_peft_model(model, cfg)
trainable = sum(p.numel() for p in peft_model.parameters() if p.requires_grad)
print(f"trainable: {trainable/1e6:.1f} M of {n/1e6:.0f} M numbers ({100*trainable/n:.2f}%)")

examples = [example(r) for r in train.itertuples()]
order = list(range(len(examples))); random.Random(SEED).shuffle(order)
opt = torch.optim.AdamW([p for p in peft_model.parameters() if p.requires_grad], lr=2e-4)
peft_model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
peft_model.enable_input_require_grads(); model.config.use_cache = False
peft_model.train(); t0 = time.time(); losses = []
for step, i in enumerate(range(0, len(order), 8)):
    batch = [examples[j] for j in order[i:i + 8]]
    L = max(len(a) for a, _ in batch)
    ids = torch.tensor([a + [tok.pad_token_id] * (L - len(a)) for a, _ in batch]).to(device)
    lab = torch.tensor([b + [-100] * (L - len(b)) for _, b in batch]).to(device)
    att = torch.tensor([[1] * len(a) + [0] * (L - len(a)) for a, _ in batch]).to(device)
    loss = peft_model(input_ids=ids, attention_mask=att, labels=lab).loss
    loss.backward(); opt.step(); opt.zero_grad(set_to_none=True); losses.append(loss.item())
    if step % 15 == 0: print(f"step {step:>3}  loss {loss.item():.3f}")
peft_model.gradient_checkpointing_disable(); model.config.use_cache = True; peft_model.eval()
print(f"{len(losses)} steps in {time.time() - t0:.0f} s on {device}; loss {losses[0]:.2f} -> {np.mean(losses[-5:]):.3f}")


## A.5 — Measure it honestly

Same 40 held-out tickets, same scorer, same minimal prompt. The only thing that changed is the adapter.


In [ ]:
tuned_raw = generate(peft_model, list(EVAL.text))
tuned_exact = score(tuned_raw, "fine-tuned + minimal prompt")
print(f"\nadapter off -> on, all four right: {base_exact} -> {tuned_exact} of {len(EVAL)}")
for t, r in list(zip(EVAL.text, tuned_raw))[:2]:
    print(f"\n{t[:110]}\n -> {r[:120]}")


Read the field columns, not only the headline. Format and the queue are cheap to teach; the
page-a-person flag is a four-condition policy, and 720 examples on a half-billion-number model is a
thin way to teach judgement. If the tool goes live, compute any escalation rule you can in code from
the fields the model gets right, and keep a person on the rest. That is a **hard guardrail**, and the
Streamlit page in Part C already does the first half of it.

## A.6 — Save the adapter and download it

What you trained is only the adapter: two thin tables per weight table, about 35 MB. That is what
travels to your laptop. (The base model is public; Part B downloads its own copy.)


In [ ]:
peft_model.save_pretrained("ism6564-week06-adapter")
shutil.make_archive("ism6564-week06-adapter", "zip", "ism6564-week06-adapter")
print(f"saved ism6564-week06-adapter.zip ({os.path.getsize('ism6564-week06-adapter.zip') / 1e6:.0f} MB)")
if IN_COLAB:
    from google.colab import files
    files.download("ism6564-week06-adapter.zip")          # lands in your browser's Downloads folder
else:
    print("not on Colab: the zip is in this folder")


---

# Part B — on your computer: serve the model with Ollama

You now have `ism6564-week06-adapter.zip` on your machine. Three steps, none of which needs a GPU:

1. **Install Ollama** if you have not: [ollama.com/download](https://ollama.com/download). Check it
   runs: `ollama list` in a terminal.
2. **Convert the adapter into an Ollama model.** Put the zip in a folder with
   `ism6564-week06-export-to-ollama.ipynb` (Canvas) and run that notebook top to bottom in the course
   environment. It downloads the base model, folds your adapter into it, converts the result into one
   GGUF file (the format Ollama runs), and runs `ollama create ism6564-w6-triage` for you. About two
   minutes; about 2.5 GB of disk while it works.
3. **Check it is there.** In a terminal, `ollama list` should now show `ism6564-w6-triage`. (Do not
   judge it with `ollama run` and a bare ticket: that sends no system prompt, and the minimal prompt the
   adapter was trained with is what tells it to answer with the object. Part C sends the prompt.)

Ollama now serves your model through the **same API shape as OpenAI** at
`http://localhost:11434/v1/`. Anything written against the OpenAI client works by changing two
settings: the `base_url`, and any string as the key (Ollama ignores it, the client insists on one).
That is what Part C uses.

**Why GGUF, and why not just `ollama pull`?** Ollama's library and Hugging Face both hold ready-made
GGUF models you can pull with one command (`ollama pull qwen2.5:0.5b`, or `ollama run
hf.co/<user>/<repo>-GGUF`). Those are *base* or *generally tuned* models. Yours has Northwind's codes and
policy in its weights, so it has to be converted from your adapter; the export notebook does that.

---

# Part C — on your computer: a Streamlit page to test it

`ism6564-week06-workflow-app.py` (Canvas; the cell below also writes a copy next to this notebook) is
about eighty lines. It is worth reading in full, because it is the shape of every small LLM tool:

- **a client** pointed at Ollama (`OpenAI(base_url=..., api_key="ollama")`);
- **the same minimal prompt** the adapter was trained with (change the prompt and you change the
  behaviour you measured);
- **the contract**, as a Pydantic model with `StrictBool`, and a `check()` that never raises;
- **the page**: a text box, a button, the object on the left, the raw reply on the right, the time and
  token count underneath; an invalid reply is shown as *routed to a person*, not as an error.

Run it (the course environment plus `pip install streamlit openai pydantic`):

```
streamlit run ism6564-week06-workflow-app.py
```

A browser tab opens at `http://localhost:8501`. Paste a ticket, press **Triage**. Then try the
un-tuned base model for comparison: `ollama pull qwen2.5:0.5b` once, and type `qwen2.5:0.5b` in the
sidebar. Same prompt, same page: the difference is what 720 labelled tickets taught the adapter.


In [ ]:
APP_SOURCE = r'''"""ISM 6564 · Week 6 · Workflow example, Part C: a Streamlit page that tests YOUR fine-tuned model.

The model runs in Ollama on this machine (Part B). This page is a thin client: it sends the ticket to
Ollama through the OpenAI-compatible API, checks the reply against the contract, and shows the result.
Nothing here needs PyTorch or a GPU.

    pip install streamlit openai pydantic
    streamlit run ism6564-week06-workflow-app.py
"""
import json
import re
import time
from typing import Literal

import streamlit as st
from openai import OpenAI
from pydantic import BaseModel, Field, StrictBool, ValidationError

OLLAMA_URL = "http://localhost:11434/v1/"
DEFAULT_MODEL = "ism6564-w6-triage"          # created by ism6564-week06-export-to-ollama.ipynb
CODES = ["SHIP-DELAY", "RET-RMA", "BILL-DISP", "PROD-DEF", "ACCT-SEC", "PROD-INFO"]

# the SAME minimal prompt the adapter was trained with (Part A) -- the codes' meanings and the policy
# live in the weights now, so the prompt only names the fields
SYSTEM = ("You triage support tickets for Northwind Outfitters.\n"
          'Reply with JSON: {"route_to": one of ' + "|".join(CODES) +
          ', "urgency": low|normal|high, "needs_human": true|false, "order_id": string or ""}')


class Triage(BaseModel):
    """The contract (lecture Block 4): a hard guardrail. StrictBool rejects the text "no"."""
    route_to: Literal["SHIP-DELAY", "RET-RMA", "BILL-DISP", "PROD-DEF", "ACCT-SEC", "PROD-INFO"]
    urgency: Literal["low", "normal", "high"]
    needs_human: StrictBool
    order_id: str = Field(default="", pattern=r"^(NW-\d{5})?$")


def call_model(client: OpenAI, model: str, ticket: str) -> tuple[str, float, int]:
    t0 = time.time()
    resp = client.chat.completions.create(
        model=model, temperature=0, max_tokens=80,
        messages=[{"role": "system", "content": SYSTEM},
                  {"role": "user", "content": f"<ticket>\n{ticket}\n</ticket>"}],
    )
    text = resp.choices[0].message.content or ""
    return text, time.time() - t0, (resp.usage.prompt_tokens if resp.usage else 0)


def check(raw: str) -> tuple[Triage | None, str]:
    """Never raises: (object, "") on success, (None, reason) otherwise."""
    m = re.search(r"\{.*?\}", raw, re.S)
    if not m:
        return None, "no JSON object in the reply"
    try:
        return Triage.model_validate_json(m.group(0)), ""
    except ValidationError as e:
        return None, "; ".join(f"{'.'.join(map(str, x['loc']))}: {x['msg']}" for x in e.errors())


# ------------------------------------------------------------------------------------------- the page
st.set_page_config(page_title="Northwind triage", page_icon=":ticket:", layout="centered")
st.title("Northwind ticket triage")
st.caption("ISM 6564 · Week 6 workflow example · a fine-tuned Qwen2.5-0.5B served by Ollama on this machine")

with st.sidebar:
    st.header("Model")
    model = st.text_input("Ollama model", DEFAULT_MODEL)
    base_url = st.text_input("Ollama URL", OLLAMA_URL)
    st.markdown("Compare with the **un-tuned** base model: pull it once with "
                "`ollama pull qwen2.5:0.5b` and type `qwen2.5:0.5b` above.")

client = OpenAI(base_url=base_url, api_key="ollama")      # Ollama ignores the key; the client requires one

EXAMPLES = {
    "(type your own)": "",
    "Late parcel": "Order NW-40318 was due Monday and still hasn't arrived. I need it before Friday for a trip.",
    "Angry billing": "You charged my card twice for order NW-51002 and nobody answers the phone. This is unacceptable.",
    "Account takeover": "I got an email that my password was changed but I didn't do it. Someone is in my account.",
    "Product question": "Does the Ridgeline jacket run large? Thinking about a medium. No rush.",
}
pick = st.selectbox("Try a sample ticket", list(EXAMPLES))
ticket = st.text_area("Ticket", EXAMPLES[pick], height=140, placeholder="Paste a customer email here")

if st.button("Triage", type="primary", disabled=not ticket.strip()):
    try:
        raw, secs, in_tok = call_model(client, model, ticket.strip())
    except Exception as e:                       # Ollama not running, model not created, wrong name ...
        st.error(f"Could not reach the model: {e}")
        st.info("Is Ollama running (`ollama list`)? Did Part B create the model (`ollama create ...`)?")
        st.stop()

    obj, why = check(raw)
    left, right = st.columns([1.1, 0.9])
    with left:
        st.subheader("The object")
        if obj is None:
            # the hard guardrail: an invalid reply never reaches the rest of the system
            st.error(f"Reply failed the contract: {why}")
            st.warning("Routed to a person.  (This is the guardrail doing its job, not a crash.)")
        else:
            st.json(obj.model_dump())
            if obj.needs_human:
                st.warning("needs_human = true: a person is paged.")
            else:
                st.success(f"Queue {obj.route_to}, urgency {obj.urgency}.")
    with right:
        st.subheader("What the model wrote")
        st.code(raw or "(empty reply)", language="json")
        st.caption(f"{secs:.2f} s · {in_tok} prompt tokens · model `{model}`")

st.divider()
st.markdown(
    "**Read the result the way the lecture does.** *Valid* is not *right*: the contract check only says the "
    "object is well-formed. Whether the queue and the escalation flag are right is what Part A measured on "
    "held-out tickets. Facts the model cannot know (where the parcel is) are not in this object at all; "
    "that is Week 7."
)
'''
open('ism6564-week06-workflow-app.py', 'w', encoding='utf-8').write(APP_SOURCE)
print(f"wrote ism6564-week06-workflow-app.py ({len(APP_SOURCE.splitlines())} lines) -- run it with:  streamlit run ism6564-week06-workflow-app.py")


## What would make this a product, and which week covers it

| Gap | Week |
|---|---|
| It knows nothing about *this* customer's order, stock or the current returns window | **Week 7**: retrieve the facts into the window at call time (RAG), and manage a window that fills |
| It only produces an object; it cannot look the order up or issue a refund | **Week 8**: tools and agents, which call exactly this kind of contract |
| Several such models and tools have to work together | **Week 9** |
| The page-a-person flag is the weakest field, and nobody has red-teamed the input box | **Week 10**: guardrails tested, evaluation as a release gate, red-teaming |

And the one that is already true: the customer's text never left the machine, the model costs nothing
per call, and every reply is checked against a contract before anything acts on it.
